# my-garmin – áttekintés
Az adatok a `data/garmin.db` SQLite-ból és a `data/raw` nyers fájlokból jönnek.
Frissítés: `mygarmin sync` (a projekt gyökeréből).

In [ ]:
import json, sqlite3, zipfile, io
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path.cwd().parent / "data" if Path.cwd().name == "notebooks" else Path.cwd() / "data"
conn = sqlite3.connect(DATA / "garmin.db")


def plot(df, cols, ax, title):
    cols = [c for c in cols if df[c].notna().any()]
    if cols:
        df[cols].plot(ax=ax, title=title)
    else:
        ax.set_title(f"{title} (nincs adat)")


pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)

## Napi összesítő: lépés, nyugalmi pulzus, stressz, Body Battery

In [ ]:
daily = pd.read_sql("SELECT * FROM daily_summary ORDER BY date", conn, parse_dates=["date"]).set_index("date")
fig, ax = plt.subplots(4, 1, figsize=(12, 10), sharex=True)
plot(daily, ["steps"], ax[0], "Lépés")
plot(daily, ["resting_hr"], ax[1], "Nyugalmi pulzus")
plot(daily, ["avg_stress"], ax[2], "Átlagos stressz")
plot(daily, ["body_battery_high", "body_battery_low"], ax[3], "Body Battery")
plt.tight_layout()
daily.tail()

## Alvás fázisok

In [ ]:
sleep = pd.read_sql("SELECT * FROM sleep ORDER BY date", conn, parse_dates=["date"]).set_index("date")
phases = (sleep[["deep_s", "light_s", "rem_s", "awake_s"]].fillna(0) / 3600)
phases.index = phases.index.date
phases.plot(kind="bar", stacked=True, figsize=(12, 4), title="Alvás (óra)")
sleep[["score"]].describe()

## Aktivitások

In [ ]:
acts = pd.read_sql("SELECT * FROM activities ORDER BY start_local DESC", conn)
acts.groupby("type").agg(db=("activity_id", "count"), ora=("duration_s", lambda s: s.sum() / 3600)).sort_values("db", ascending=False)

## Nyers adat példa: napon belüli pulzus (raw JSON)
Amit még nem töltünk DB-be, az is elérhető a `data/raw` alatt.

In [ ]:
hr_files = sorted((DATA / "raw" / "daily" / "get_heart_rates").glob("*.json"))
hr = json.loads(hr_files[-1].read_text(encoding="utf-8"))
values = pd.DataFrame(hr.get("heartRateValues") or [], columns=["ts", "bpm"])
values["ts"] = pd.to_datetime(values["ts"], unit="ms")
values.set_index("ts")["bpm"].plot(figsize=(12, 3), title=f"Pulzus – {hr_files[-1].stem}")

## Nyers adat példa: aktivitás eredeti FIT fájlja

In [ ]:
from fitparse import FitFile
act_dir = Path(acts.iloc[0]["raw_dir"])
with zipfile.ZipFile(act_dir / "original.zip") as z:
    fit_name = next(n for n in z.namelist() if n.lower().endswith(".fit"))
    fit = FitFile(io.BytesIO(z.read(fit_name)))
records = pd.DataFrame([{f.name: f.value for f in r} for r in fit.get_messages("record")])
records.head()